# 01 — Data Collection and Cleaning

This notebook loads the BNR Excel export and prepares a clean official USD/RWF observation dataset.

Raw file:

`data/raw/currency_table.xlsx`

Expected raw columns from the BNR export:
- `currency_name`
- `buying_rate`
- `average_rate`
- `selling_rate`
- `post_date`

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent
RAW_FILE = ROOT / 'data' / 'raw' / 'currency_table.xlsx'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE

WindowsPath('c:/Users/Prime core/Downloads/FXGuard AI/FXGuard_AI_Project/data/raw/currency_table.xlsx')

In [4]:
raw = pd.read_excel(RAW_FILE)
raw.head()

,currency_name,buying_rate,average_rate,selling_rate,post_date
0,USD,"1,008.49","1,018.58","1,028.66",01-Apr-22
1,USD,1401.182055,1415.19246,1429.202865,01-Apr-25
2,USD,1455.535,1460.535,1465.535,01-Apr-26
3,USD,"1,018.88","1,029.07","1,039.26",01-Aug-22
4,USD,"1,167.32","1,178.99","1,190.66",01-Aug-23


In [5]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   currency_name  1096 non-null   str  
 1   buying_rate    1096 non-null   str  
 2   average_rate   1096 non-null   str  
 3   selling_rate   1096 non-null   str  
 4   post_date      1096 non-null   str  
dtypes: str(5)
memory usage: 42.9 KB


## Clean numeric columns

The raw BNR export may contain values with commas such as `1,008.49`. The function below converts these into numeric values.

In [6]:
def to_number(value):
    if pd.isna(value):
        return np.nan
    return float(str(value).replace(',', '').strip())

clean = raw.copy()
clean = clean.rename(columns={
    'currency_name': 'currency',
    'post_date': 'date'
})

clean['date'] = pd.to_datetime(clean['date'], errors='coerce', dayfirst=False)
for col in ['buying_rate', 'average_rate', 'selling_rate']:
    clean[col] = clean[col].apply(to_number)

clean['currency'] = clean['currency'].astype(str).str.upper().str.strip()
clean = clean.dropna(subset=['date', 'currency', 'buying_rate', 'average_rate', 'selling_rate'])
clean = clean[clean['currency'] == 'USD'].copy()
clean['mid_rate'] = clean['average_rate']
clean = clean.drop_duplicates(subset=['date', 'currency']).sort_values('date').reset_index(drop=True)

clean.head(), clean.tail(), clean.shape

C:\Users\Prime core\AppData\Local\Temp\ipykernel_12128\3038077829.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  clean['date'] = pd.to_datetime(clean['date'], errors='coerce', dayfirst=False)


(  currency  buying_rate  average_rate  selling_rate       date  mid_rate
 0      USD   999.820796       1009.82       1019.82 2022-01-04   1009.82
 1      USD   999.920017       1009.92       1019.92 2022-01-05   1009.92
 2      USD  1000.100000       1010.10       1020.10 2022-01-06   1010.10
 3      USD  1000.200000       1010.20       1020.20 2022-01-07   1010.20
 4      USD  1000.330000       1010.33       1020.33 2022-01-10   1010.33,
      currency  buying_rate  average_rate  selling_rate       date  mid_rate
 1091      USD     1460.070      1465.070      1470.070 2026-06-19  1465.070
 1092      USD     1460.165      1465.165      1470.165 2026-06-22  1465.165
 1093      USD     1460.260      1465.260      1470.260 2026-06-23  1465.260
 1094      USD     1460.350      1465.350      1470.350 2026-06-24  1465.350
 1095      USD     1460.445      1465.445      1470.445 2026-06-25  1465.445,
 (1096, 6))

## Keep the full 4-year period from the Excel file

The prepared dataset uses the available 4-year window ending on the latest date in the Excel file. It creates:

1. official BNR observations only
2. daily calendar version with weekends/non-posting days forward-filled

In [7]:
latest_date = clean['date'].max()
start_date = latest_date - pd.DateOffset(years=4) + pd.DateOffset(days=1)
print('Start:', start_date.date())
print('End:', latest_date.date())

clean_4yr = clean[(clean['date'] >= start_date) & (clean['date'] <= latest_date)].copy()
clean_4yr.to_csv(PROCESSED_DIR / 'bnr_official_observations_4year.csv', index=False)
print(clean_4yr.shape)
clean_4yr.head()

Start: 2022-06-26
End: 2026-06-25
(978, 6)


,currency,buying_rate,average_rate,selling_rate,date,mid_rate
118,USD,1013.97,1024.10,1034.24,2022-06-27,1024.10
119,USD,1013.77,1023.90,1034.04,2022-06-28,1023.90
120,USD,1014.06,1024.20,1034.34,2022-06-29,1024.20
121,USD,1014.34,1024.48,1034.62,2022-06-30,1024.48
122,USD,1014.48,1024.62,1034.77,2022-07-05,1024.62


In [8]:
# Build continuous daily calendar. Official BNR rates are not always posted on weekends/holidays.
# For importer decision support, we forward-fill the latest available official rate.

daily_dates = pd.date_range(start=start_date.normalize(), end=latest_date.normalize(), freq='D')
daily = pd.DataFrame({'date': daily_dates})
daily['currency'] = 'USD'

daily = daily.merge(clean_4yr, on=['date', 'currency'], how='left', indicator=True)
daily['is_official_observation'] = (daily['_merge'] == 'both').astype(int)
daily = daily.drop(columns=['_merge'])

for col in ['buying_rate', 'average_rate', 'selling_rate', 'mid_rate']:
    daily[col] = daily[col].ffill().bfill()

daily.to_csv(PROCESSED_DIR / 'exchange_rates_daily_calendar_4year.csv', index=False)
print(daily.shape)
daily.head(), daily.tail()

(1461, 7)


(        date currency  buying_rate  average_rate  selling_rate  mid_rate  \
 0 2022-06-26      USD      1013.97       1024.10       1034.24   1024.10   
 1 2022-06-27      USD      1013.97       1024.10       1034.24   1024.10   
 2 2022-06-28      USD      1013.77       1023.90       1034.04   1023.90   
 3 2022-06-29      USD      1014.06       1024.20       1034.34   1024.20   
 4 2022-06-30      USD      1014.34       1024.48       1034.62   1024.48   
 
    is_official_observation  
 0                        0  
 1                        1  
 2                        1  
 3                        1  
 4                        1  ,
            date currency  buying_rate  average_rate  selling_rate  mid_rate  \
 1456 2026-06-21      USD     1460.070      1465.070      1470.070  1465.070   
 1457 2026-06-22      USD     1460.165      1465.165      1470.165  1465.165   
 1458 2026-06-23      USD     1460.260      1465.260      1470.260  1465.260   
 1459 2026-06-24      USD     1460.

In [9]:
print('Official observations:', int(daily['is_official_observation'].sum()))
print('Daily calendar rows:', len(daily))
print('Missing values:')
print(daily.isna().sum())

Official observations: 978
Daily calendar rows: 1461
Missing values:
date                       0
currency                   0
buying_rate                0
average_rate               0
selling_rate               0
mid_rate                   0
is_official_observation    0
dtype: int64
